# Model 1 pipeline (MedGemma 4B, no reasoning)

Predicts the discharge diagnoses for each admission from the cleaned patient record.
Scoring is done in `judging.ipynb`.

**Before running:** the vLLM server must be running with MedGemma, e.g.
`vllm serve google/medgemma-4b-it --dtype bfloat16 --max-model-len 131072 --max-num-batched-tokens 4096 --gpu-memory-utilization 0.90 --host 127.0.0.1`

**Run order:** top to bottom. `predictions_model1.json` is already complete (99 cases), so section 6 will report "nothing to do" unless you delete it.

**Privacy:** saved outputs of this notebook may contain patient-derived text. Clear all outputs before sharing it.

## 1. Configuration

In [ ]:
from pathlib import Path

# --- predictor, served by vLLM ---
#MODEL_ID   = "google/medgemma-4b-it"
MODEL_ID  = "bbarn4/medgemma-27b-text-it-GPTQ"
SERVER_URL = "http://localhost:8000/v1"

# --- data locations (relative to this notebook) ---
PATIENT_DIR = Path("patient_contexts")
CHUNKS_FILE = Path("all_patient_chunks.jsonl")
DEV_CASE    = "patient_00000000_00000000.txt"   # tuning case, excluded from the test set

# --- outputs ---
#PRED_FILE        = Path("predictions_model1.json")
PRED_FILE = Path("predictions_model1_27b.json")
CLEAN_TOKENS_CSV = Path("cleaned_token_counts.csv")

# --- generation settings (fixed for every run, no change after seeing results) ---
MAX_NEW_TOKENS     = 600
REPETITION_PENALTY = 1.05

## 2. Tokenizer and server connection

In [ ]:
from huggingface_hub import login
from transformers import AutoTokenizer

login()   # paste your Hugging Face token when asked
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("Tokenizer loaded:", type(tokenizer).__name__)

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=SERVER_URL, api_key="x")
served = [m.id for m in client.models.list().data]
assert MODEL_ID in served, f"The server is serving {served}, not {MODEL_ID}. Start the correct vLLM server first."
print("Server OK, serving:", served)

## 3. Input cleaning (code only, no LLM)

- **INPUT EVENTS:** one line per drug/fluid (entries, first and last time, total amount, rate range).
- **MEDICATIONS + MEDICATION ADMINISTRATION:** merged into one inpatient medication summary (one line per drug).
- Everything else (labs, vitals, HPI, radiology, ECG ...) is left unchanged.

In [ ]:
import io, re
import pandas as pd

def parse_fwf_table(text, first_col):
    frames, header, block = [], None, []

    def flush():
        if header is None or not block:
            return
        width = max(len(x.rstrip()) for x in [header] + block)
        hdr = header.rstrip().rjust(width)
        rows = [x.rstrip().rjust(width) for x in block]
        toks = list(re.finditer(r"\S+", hdr))
        ends = [t.end() for t in toks]
        specs = [(0 if i == 0 else ends[i-1], ends[i]) for i in range(len(ends))]
        names = [t.group() for t in toks]
        frames.append(pd.read_fwf(io.StringIO("\n".join(rows)), colspecs=specs, names=names,
                                  header=None, dtype=str, keep_default_na=False))

    for line in text.splitlines():
        toks = line.split()
        if toks and toks[0] == first_col and len(toks) > 3:
            flush()
            header, block = line, []
            continue
        if header is None or not line.strip() or set(line.strip()) <= set("=-"):
            continue
        block.append(line)
    flush()
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def _num(s):
    return pd.to_numeric(s.replace("NaN", pd.NA), errors="coerce")

def aggregate_input_events(section_text, exclude_status=()):
    df = parse_fwf_table(section_text, "starttime")
    if df.empty:
        return section_text, {"parsed": 0}
    df = df[~df["statusdescription"].isin(exclude_status)].copy()
    df["start"] = pd.to_datetime(df["starttime"], errors="coerce")
    df["end"] = pd.to_datetime(df["endtime"], errors="coerce")
    df["amt"], df["rt"] = _num(df["amount"]), _num(df["rate"])
    bad = int(df["start"].isna().sum())
    lines = []
    for (label, cat, uom), g in df.groupby(["label", "ordercategoryname", "amountuom"], sort=False):
        first, last = g["start"].min(), g["end"].max()
        s = f"- {label} [{cat[3:] if cat[:3].strip('0123456789') == '-' else cat}]: {len(g)} entr{'y' if len(g)==1 else 'ies'}, {first:%Y-%m-%d %H:%M} to {last:%Y-%m-%d %H:%M}"
        if g["amt"].notna().any() and uom != "NaN":
            s += f", total {g['amt'].sum():.4g} {uom}"
        ru = [u for u in g["rateuom"] if u != "NaN"]
        if g["rt"].notna().any() and ru:
            s += f", rate {g['rt'].min():.4g}-{g['rt'].max():.4g} {ru[0]}"
        lines.append((first, s))
    lines.sort(key=lambda x: x[0])
    header = "INPUT EVENTS SUMMARY (inpatient infusions, boluses and blood products; NOT discharge medications):"
    return header + "\n" + "\n".join(s for _, s in lines), {"parsed": len(df), "bad_dates": bad}

def _dose_range(series):
    vals = []
    for x in series:
        vals += [float(v) for v in re.findall(r"\d+(?:\.\d+)?", str(x))]
    if not vals:
        return ""
    lo, hi = min(vals), max(vals)
    return f"{lo:g}" if lo == hi else f"{lo:g}-{hi:g}"

def _med_table(text, first_col):
    df = parse_fwf_table(text, first_col)
    if df.empty:
        return df
    df.columns = ["drug", "dose", "unit", "route", "first", "last", "n"]
    df["key"] = df["drug"].str.strip().str.lower()
    df["n"] = pd.to_numeric(df["n"], errors="coerce").fillna(0).astype(int)
    df["first"] = pd.to_datetime(df["first"], errors="coerce")
    df["last"] = pd.to_datetime(df["last"], errors="coerce")
    return df

def merge_medication_sections(meds_text, admin_text):
    rx, ad = _med_table(meds_text, "drug"), _med_table(admin_text, "medication")
    info = {"rx_rows": len(rx), "admin_rows": len(ad)}
    if rx.empty and ad.empty:
        return None, info
    keys = sorted(set(rx.get("key", [])) | set(ad.get("key", [])))
    out = []
    for k in keys:
        r = rx[rx["key"] == k] if not rx.empty else rx
        a = ad[ad["key"] == k] if not ad.empty else ad
        name = (r["drug"].iloc[0] if len(r) else a["drug"].iloc[0]).strip()
        parts = []
        for label, d in (("prescribed", r), ("administered", a)):
            if len(d):
                unit = next((u for u in d["unit"] if u != "NaN"), "")
                dr = _dose_range(d["dose"])
                parts.append(f"{label} {d['n'].sum()}x" + (f", dose {dr} {unit}".rstrip() if dr else ""))
        routes = sorted({x.strip() for d in (r, a) if len(d) for x in d["route"]} - {"NaN", ""})
        if routes:
            parts.append("route " + "/".join(routes))
        lasts = [d["last"].max() for d in (r, a) if len(d)]
        firsts = [d["first"].min() for d in (r, a) if len(d)]
        last, first = max(lasts), min(firsts)
        out.append((last, f"- {name}: " + "; ".join(parts) + f"; {first:%Y-%m-%d} to {last:%Y-%m-%d %H:%M}"))
    out.sort(key=lambda x: x[0], reverse=True)
    header = "INPATIENT MEDICATION SUMMARY (prescriptions + administrations, most recently active first; NOT discharge medications):"
    return header + "\n" + "\n".join(s for _, s in out), info

In [ ]:
import json
from collections import defaultdict

def load_patient_sections(subject_id, hadm_id, tokenizer, chunks_file=CHUNKS_FILE):
    """Rebuild one admission at section level, applying the two cleaning steps."""
    sections = defaultdict(list)
    with open(chunks_file, "r", encoding="utf-8") as f:
        for line in f:
            chunk = json.loads(line)
            if str(chunk["subject_id"]) == str(subject_id) and str(chunk["hadm_id"]) == str(hadm_id):
                sections[chunk["section"]].append(chunk)
    if not sections:
        raise ValueError(f"No chunks found for subject_id={subject_id}, hadm_id={hadm_id}")

    patient_sections = []
    for section, section_chunks in sections.items():
        section_chunks.sort(key=lambda x: x.get("chunk_id", 0))
        section_text = "\n\n".join(c["text"].strip() for c in section_chunks if c["text"].strip())
        if section == "INPUT EVENTS":
            section_text, info = aggregate_input_events(section_text)
            print("INPUT EVENTS aggregation:", info)
        patient_sections.append({
            "section": section,
            "chunk_ids": [c.get("chunk_id") for c in section_chunks],
            "text": section_text,
            "token_count": len(tokenizer.encode(section_text, add_special_tokens=False)),
        })

    by_name = {s["section"]: s for s in patient_sections}
    if "MEDICATIONS" in by_name and "MEDICATION ADMINISTRATION" in by_name:
        merged, info = merge_medication_sections(by_name["MEDICATIONS"]["text"],
                                                 by_name["MEDICATION ADMINISTRATION"]["text"])
        print("Medication merge:", info)
        if merged:
            by_name["MEDICATIONS"]["text"] = merged
            by_name["MEDICATIONS"]["token_count"] = len(tokenizer.encode(merged, add_special_tokens=False))
            by_name["MEDICATION ADMINISTRATION"]["text"] = ""
            by_name["MEDICATION ADMINISTRATION"]["token_count"] = 0
    return patient_sections

# Helpers used everywhere below
import contextlib
CASE_RE = re.compile(r"patient_(\d+)_(\d+)\.txt$")

def get_sections(fname):
    m = CASE_RE.match(fname)
    with contextlib.redirect_stdout(io.StringIO()):      # silence the per-case info lines
        return load_patient_sections(m.group(1), m.group(2), tokenizer)

def build_patient_text(secs):
    return "\n\n".join(f"{s['section']}\n{s['text'].strip()}" for s in secs if s["text"].strip())

In [ ]:
# ============================================================
# Build all_patient_chunks_v2.jsonl
# Unchanged sections: original chunks kept as-is.
# Cleaned sections: Model 1's cleaned text, re-chunked with the
# same table_chunk logic as 3-Patient_Chunking_Pipeline.
# ============================================================
import json, io, contextlib
from collections import defaultdict, Counter
from transformers import AutoTokenizer

OLD_CHUNKS = Path("all_patient_chunks.jsonl")
NEW_CHUNKS = Path("all_patient_chunks_v2.jsonl")
CLEANED = {"INPUT EVENTS", "MEDICATIONS", "MEDICATION ADMINISTRATION"}

bge_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")
def bge_count(t):
    return len(bge_tokenizer.encode(t, add_special_tokens=False))

def table_chunk_v2(text, max_tokens=480):
    """Same logic as table_chunk: first line is the header, repeated in every chunk."""
    lines = text.splitlines()
    if len(lines) < 2:
        return []
    header = lines[0].strip()
    rows = [r.strip() for r in lines[1:] if r.strip()]
    h = bge_count(header)
    out, cur, n = [], [], h
    for r in rows:
        t = bge_count(r)
        if n + t <= max_tokens:
            cur.append(r); n += t
        elif not cur:
            out.append("\n".join([header, r]))      # single oversized row
        else:
            out.append("\n".join([header] + cur))
            cur, n = [r], h + t
    if cur:
        out.append("\n".join([header] + cur))
    return out

# Load old chunks grouped by admission, keeping file order
old, order = defaultdict(list), []
with open(OLD_CHUNKS, encoding="utf-8") as f:
    for line in f:
        c = json.loads(line)
        key = (str(c["subject_id"]), str(c["hadm_id"]))
        if key not in old:
            order.append(key)
        old[key].append(c)

new_all = []
for sid, hid in order:
    chunks = sorted(old[(sid, hid)], key=lambda c: c["chunk_id"])

    # Section text exactly as Model 1 rebuilt it, before and after cleaning
    by_sec = defaultdict(list)
    for c in chunks:
        by_sec[c["section"]].append(c)
    orig_text = {s: "\n\n".join(c["text"].strip() for c in cs if c["text"].strip())
                 for s, cs in by_sec.items()}
    with contextlib.redirect_stdout(io.StringIO()):
        cleaned = {s["section"]: s["text"]
                   for s in load_patient_sections(sid, hid, tokenizer, chunks_file=OLD_CHUNKS)}

    new_chunks, done = [], set()
    for c in chunks:
        sec = c["section"]
        changed = sec in CLEANED and cleaned.get(sec, "") != orig_text[sec]
        if not changed:
            new_chunks.append({k: v for k, v in c.items() if k != "chunk_id"})
            continue
        if sec in done:
            continue
        done.add(sec)
        for t in table_chunk_v2(cleaned.get(sec, "")):
            new_chunks.append({"subject_id": c["subject_id"], "hadm_id": c["hadm_id"],
                               "section": sec, "chunk_type": "table",
                               "parent_id": sec, "text": t})

    for i, c in enumerate(new_chunks, start=1):
        c["chunk_id"] = i
    new_all.extend(new_chunks)

# Checks
oldc = Counter(c["section"] for k in order for c in old[k])
newc = Counter(c["section"] for c in new_all)
print("Admissions:", len(order))
print(f"Chunks: {sum(oldc.values()):,} -> {len(new_all):,}")
for sec in sorted(CLEANED):
    print(f"  {sec:28s} {oldc[sec]:>7,} -> {newc[sec]:>6,}")
print("Unchanged sections have identical chunk counts:",
      all(oldc[s] == newc[s] for s in oldc if s not in CLEANED))
print("Max BGE tokens in a re-chunked chunk:",
      max((bge_count(c["text"]) for c in new_all if c["section"] in CLEANED), default=0))

with open(NEW_CHUNKS, "w", encoding="utf-8") as f:
    for c in new_all:
        f.write(json.dumps(c) + "\n")
print("Saved:", NEW_CHUNKS)

## 4. Cohort and cleaned token counts

In [ ]:
case_files = sorted(p.name for p in PATIENT_DIR.glob("patient_*.txt") if CASE_RE.match(p.name))

if CLEAN_TOKENS_CSV.exists():
    df_clean = pd.read_csv(CLEAN_TOKENS_CSV)
    print("Loaded cached counts from", CLEAN_TOKENS_CSV)
else:
    rows = [{"file_name": f, "tokens": sum(s["token_count"] for s in get_sections(f))} for f in case_files]
    df_clean = pd.DataFrame(rows)
    df_clean.to_csv(CLEAN_TOKENS_CSV, index=False)
    print("Computed and saved", CLEAN_TOKENS_CSV)

assert set(df_clean["file_name"]) == set(case_files), "Cached token counts do not match the files in PATIENT_DIR - delete the CSV and re-run."
print("cases:", len(df_clean), "| excluding the development case:", len(df_clean) - (DEV_CASE in set(df_clean["file_name"])))
print(df_clean["tokens"].describe().round(0).to_string())
for t in (20_000, 30_000, 58_000, 120_000):
    print(f"cases <= {t:,} tokens: {(df_clean['tokens'] <= t).sum()} of {len(df_clean)}")

## 5. Prompt

One ordered list of diagnoses (at most 15), instructions placed **after** the record.
The prompt is fixed: do not edit it after seeing scores. Model 2 and any other predictor must use the same list format and the same cap.

In [ ]:
MODEL1_PROMPT = """
You are given the available clinical record for a single hospital admission.

Using only the information in the record, predict the discharge diagnoses.

List the diagnoses as one list, most important first. Include conditions that were
actively diagnosed or treated during this admission (including new problems and
complications that developed during the stay, as shown by lab trends, medications,
procedures and ICU events) and important chronic conditions.
List each diagnosis once. At most 15 diagnoses.

Use only information supported by the record.
Do not invent unsupported clinical information.
Output only the numbered list and nothing else.

PATIENT RECORD:
"""
instructions = MODEL1_PROMPT.replace("PATIENT RECORD:", "").strip()

def build_prompt(patient_text):
    return "PATIENT RECORD:\n" + patient_text + "\n\n---\n\nThe patient record is above.\n\n" + instructions

LIST_RE = re.compile(r"^\s*(?:\d+[.)]|[-*\u2022])\s*(.+?)\s*$")

def parse_diagnoses(raw):
    return [m.group(1) for line in raw.splitlines() if (m := LIST_RE.match(line))]

def predict_case(fname):
    prompt = build_prompt(build_patient_text(get_sections(fname)))
    r = client.chat.completions.create(
        model=MODEL_ID, temperature=0, max_tokens=MAX_NEW_TOKENS,
        messages=[{"role": "user", "content": prompt}],
        extra_body={"repetition_penalty": REPETITION_PENALTY},
    )
    raw = r.choices[0].message.content
    return {"raw": raw, "diagnoses": parse_diagnoses(raw),
            "finish": r.choices[0].finish_reason, "prompt_tokens": r.usage.prompt_tokens}

## 6. Run Model 1 (resumable: results are saved after every case)

In [ ]:
# The following is for MedGemma 4B Model
# N_CASES = None      # set to e.g. 3 for a quick test; None = all remaining cases

# preds1 = json.load(open(PRED_FILE, encoding="utf-8")) if PRED_FILE.exists() else {}
# todo = [f for f in df_clean["file_name"] if f != DEV_CASE and f not in preds1]
# if N_CASES:
#     todo = todo[:N_CASES]
# print("already done:", len(preds1), "| to do:", len(todo))
# if not todo:
#     print("Nothing to do.")

# for i, fname in enumerate(todo, 1):
#     try:
#         preds1[fname] = predict_case(fname)
#     except Exception as e:
#         print("failed:", fname[:24], type(e).__name__)
#         continue
#     json.dump(preds1, open(PRED_FILE, "w", encoding="utf-8"), indent=2)
#     if i % 10 == 0:
#         print(f"{i}/{len(todo)} done")
#=============================================


# The following is for MedGemma 27B Model
import time
GT = json.load(open("ground_truth_dx.json", encoding="utf-8"))

MAX_PROMPT_TOKENS = 38_000   # server window is 32,768; after restarting with --max-model-len 40000, use 39_000
N_CASES = None                  # test run; set to None for all remaining cases that fit
NEAR_MEDIAN = False           # for the test, pick cases close to the median length (28k tokens)

preds = json.load(open(PRED_FILE, encoding="utf-8")) if PRED_FILE.exists() else {}
cand = df_clean[df_clean["file_name"].isin(GT) & (df_clean["tokens"] <= MAX_PROMPT_TOKENS)
                & ~df_clean["file_name"].isin(preds)].copy()
if NEAR_MEDIAN:
    cand = cand.assign(d=(cand["tokens"] - 28_000).abs()).sort_values("d")
todo = list(cand["file_name"])[:N_CASES] if N_CASES else list(cand["file_name"])
print("already done:", len(preds), "| to do:", len(todo))

for i, fname in enumerate(todo, 1):
    t0 = time.time()
    try:
        preds[fname] = predict_case(fname)
    except Exception as e:
        print("failed:", type(e).__name__, str(e)[:120]); continue
    json.dump(preds, open(PRED_FILE, "w", encoding="utf-8"), indent=2)
    p = preds[fname]
    print(f"case {i}: prompt_tokens={p['prompt_tokens']} | seconds={round(time.time()-t0)} | finish={p['finish']} | n_diagnoses={len(p['diagnoses'])}")

## 7. Output check (counts only)

In [ ]:
import collections
p = json.load(open(PRED_FILE, encoding="utf-8"))
n = [len(v["diagnoses"]) for v in p.values()]
print("cases:", len(p))
print("finish reasons:", dict(collections.Counter(v["finish"] for v in p.values())))
print("diagnoses per case (min/median/max):", min(n), sorted(n)[len(n)//2], max(n))
print("cases with fewer than 5 diagnoses parsed:", sum(1 for x in n if x < 5))
print("max prompt tokens:", max(v["prompt_tokens"] for v in p.values()))

## Appendix: data-quality checks (optional, for the methods chapter)

These print counts only. Clear the outputs before sharing the notebook.

In [ ]:
# A. INPUT EVENTS aggregation check: row statuses and mixed rate units
raw = defaultdict(list)
with open(CHUNKS_FILE, encoding="utf-8") as f:
    for line in f:
        c = json.loads(line)
        if c["section"] == "INPUT EVENTS":
            raw[(str(c["subject_id"]), str(c["hadm_id"]))].append(c)

status, groups, mixed = collections.Counter(), 0, 0
for key, chunks in raw.items():
    chunks.sort(key=lambda x: x.get("chunk_id", 0))
    text = "\n\n".join(c["text"].strip() for c in chunks if c["text"].strip())
    df = parse_fwf_table(text, "starttime")
    if df.empty:
        continue
    status.update(df["statusdescription"])
    for _, g in df.groupby(["label", "ordercategoryname", "amountuom"]):
        groups += 1
        if g.loc[g["rateuom"] != "NaN", "rateuom"].nunique() > 1:
            mixed += 1
print("row status counts:", status.most_common())
print("groups:", groups, "| groups with mixed rate units:", mixed)

In [ ]:
# B. Outcome-wording screen on the cleaned inputs (needs expire_flags.csv from judging.ipynb)
FLAGS_FILE = Path("expire_flags.csv")
if not FLAGS_FILE.exists():
    print("expire_flags.csv not found - run the 'death flags' cell in judging.ipynb first.")
else:
    died = dict(zip(pd.read_csv(FLAGS_FILE)["file_name"], pd.read_csv(FLAGS_FILE)["died_in_hospital"]))
    TERMS = re.compile(r"\b(expired|deceased|death|died|dies|pronounced|comfort measures|comfort care|"
                       r"hospice|DNR|DNI|withdraw\w*|terminal extubation|CMO)\b", re.I)
    hit_cases = {0: set(), 1: set()}
    sec_hits = {0: collections.Counter(), 1: collections.Counter()}
    for fname in df_clean["file_name"]:
        g = int(died[fname])
        for s in get_sections(fname):
            if TERMS.search(s["text"]):
                hit_cases[g].add(fname)
                sec_hits[g][s["section"]] += 1
    for g, name in ((1, "DIED"), (0, "SURVIVED")):
        n_g = sum(1 for f in df_clean["file_name"] if int(died[f]) == g)
        print(f"{name}: {len(hit_cases[g])} of {n_g} cases have at least one hit | sections:", sec_hits[g].most_common(5))
    print("Keyword screen only: it cannot detect indirect hints. Read hits in HPI / physical exam by eye.")

In [ ]:
p27 = json.load(open("predictions_model1_27b.json", encoding="utf-8"))
for i, (f, v) in enumerate(p27.items(), 1):
    items = [d.strip().lower() for d in v["diagnoses"]]
    uniq = list(dict.fromkeys(items))
    first_dup = next((k + 1 for k, d in enumerate(items) if d in items[:k]), None)
    print(f"case {i}: finish={v['finish']} listed={len(items)} unique={len(uniq)} first_repeat_at_position={first_dup}")